# E0 - Valid max trial step count


## Notebook Setup
This cell locates the project root, imports the E0 helper module, and defines shared paths. Run it first after opening this notebook.


In [ ]:
from pathlib import Path
from typing import Optional
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        helper = candidate / "experiments" / "master-thesis" / "notebook_helpers" / "e0.py"
        if helper.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root from the current notebook working directory.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "experiments" / "master-thesis" / "notebook_helpers"
for path in (PROJECT_ROOT, HELPER_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import importlib
import e0
importlib.reload(e0)
from e0 import (
    DEFAULT_ANALYSIS_ROOT,
    DEFAULT_RESULTS_ROOT,
    analyze_e0_results,
    build_chunked_runner_script,
    load_e0_results,
    split_sample_json_into_chunks,
)
from steve_recommender.eval_v2.experimental_prep_scripts.sample_anatomies import sample_anatomies_from_registry
from steve_recommender.eval_v2.service import DefaultEvaluationService

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"HELPER_DIR={HELPER_DIR}")
print(f"pandas={pd.__version__}")


## Experiment Overview
E0 estimates a safe maximum episode step count. It samples anatomies, chunks the work for the cluster, loads the finished runs, and summarizes which step budget is sufficient.


## Experiment Setup


### E0.1 Sample random anatomies


In [ ]:
POOL = PROJECT_ROOT / 'data' / 'anatomy_registry'
OUT = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
POOL_INDEX = POOL / 'index.json'

payload = sample_anatomies_from_registry(
    pool_path=POOL,
    n=12,
    seed=123,
    strata='none',
    sampling_method='random',
    branches=('bct', 'lcca', 'lsa'),
    workers=4,
)

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

service = DefaultEvaluationService()
rows = []
for item in payload['selected_anatomies']:
    anatomy = service.get_anatomy(record_id=item['record_id'], registry_path=POOL_INDEX)
    branches = service.list_branches(anatomy)
    rows.append(
        {
            'record_id': item['record_id'],
            'arch_type': item['arch_type'],
            'seed': item['seed'],
            'target_branches': ', '.join(branch.name for branch in branches),
            'tortuosity': item['tortuosity'],
            'stratum': item['stratum'],
        }
    )

table_lines = [
    '| record_id | arch_type | seed | target_branches | tortuosity | stratum |',
    '|---|---|---:|---|---:|---|',
]
for row in rows:
    table_lines.append(
        '| {record_id} | {arch_type} | {seed} | {target_branches} | {tortuosity:.6f} | {stratum} |'.format(**row)
    )

display(Markdown('\n'.join(table_lines)))
print(json.dumps(rows, indent=2, sort_keys=True))


### E0.2 How the Chunks are created

In [ ]:
from IPython.display import Markdown, display

SAMPLE_JSON = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
rows = split_sample_json_into_chunks(SAMPLE_JSON, chunk_size=3, output_dir=SAMPLE_JSON.parent)

lines = ['| chunk_id | n_anatomies | record_ids | chunk_path |', '|---|---:|---|---|']
for row in rows:
    lines.append(
        f"| {row['chunk_id']} | {row['n_anatomies']} | {row['record_ids']} | {row['chunk_path']} |"
    )

table_md = '\n'.join(lines)
display(Markdown(table_md))


## Analysis


### Load Data Into DataFrames
The next cells load synced result files, normalize them into dataframes, and print coverage before plotting or tabulating conclusions.


### E0.3 Load data - First Dataframe

### E0.4 Analysis - Load data

This cell loads all available synced chunk results from disk and merges them into one analysis view.
Only chunks that are already present under `results/master_thesis/e0_chunks/` are included.


In [ ]:
import os

RESULT_ROOT = Path(os.environ.get('E0_RESULTS_ROOT', str(DEFAULT_RESULTS_ROOT))).resolve()
E0_CHUNK_DIR = os.environ.get('E0_CHUNK_DIR') or None
print(f'RESULT_ROOT={RESULT_ROOT}')
print(f'E0_CHUNK_DIR={E0_CHUNK_DIR}')

try:
    manifest_rows, summary_rows, trial_rows = load_e0_results(RESULT_ROOT, chunk_dir=E0_CHUNK_DIR)
except FileNotFoundError:
    display(Markdown('### No synced results found yet\n- expected root: `{RESULT_ROOT}`\n- sync the remote `results/master_thesis/e0_chunks/` tree here, preserving the per-chunk subfolders and `manifest.json` files\n- this cell always loads all available synced chunks under the result root\n- or set `E0_RESULTS_ROOT` to the synced directory before rerunning this cell'))
    raise

print(f'manifests={len(manifest_rows)}')
print(f'candidate_rows={len(summary_rows)}')
print(f'trial_rows={len(trial_rows)}')


### E0.4 Analysis - Create DataFrames

This step converts the raw lists into DataFrames so the later plots and summary tables can be built consistently.


In [ ]:
manifest_df = pd.DataFrame(manifest_rows)
summary_df = pd.DataFrame(summary_rows)
trial_df = pd.DataFrame(trial_rows)

display(Markdown('#### Run coverage (raw load)'))
display(manifest_df[['job_name', 'chunk_dir', 'n_anatomies', 'n_trials_total']].sort_values(['chunk_dir', 'job_name']))


In [ ]:
ANALYSIS_ROOT = DEFAULT_ANALYSIS_ROOT
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

if trial_df.empty:
    display(Markdown("""#### No trial rows available yet
- The synced chunk set currently contains manifests and candidate summaries, but no `trials.h5` rows.
- This is expected while only a partial chunk sync is available.
- Once more chunks are synced, rerun this cell to generate the plots and derived analysis tables."""))
    if not summary_df.empty:
        display(Markdown('#### Candidate summary rows (loaded from CSV)'))
        summary_columns = [
            column
            for column in [
                'execution_wire',
                'n_trials',
                'success_rate',
                'score_mean',
                'score_std',
                'score_safety_mean',
                'steps_total_mean',
                'steps_total_p95',
                'steps_to_success_mean',
                'steps_to_success_p95',
                'wire_force_normal_trial_max_mean_N',
                'step_total_wall_force_mean_N',
                'step_total_wall_force_max_mean_N',
            ]
            if column in summary_df.columns
        ]
        display(summary_df[summary_columns].head(20) if summary_columns else summary_df.head(20))
    else:
        display(Markdown('**No candidate summary rows were loaded either.**'))
else:
    analysis = analyze_e0_results(trial_df.to_dict('records'), analysis_root=ANALYSIS_ROOT)

    display(Markdown('#### Candidate summary'))
    display(pd.DataFrame(analysis['candidate_rows']))

    if analysis['percentiles_rows']:
        display(Markdown('#### Step-budget percentiles for successful trials'))
        percentiles_df = pd.DataFrame(analysis['percentiles_rows'])
        display(percentiles_df)
        summary_lines = [
            f"- current max_episode_steps: `{analysis['current_max']}`",
            f"- suggested 95th-percentile cutoff: `{analysis['recommended_cutoff']}` steps",
            f"- successful trials analyzed: `{len(analysis['successful_steps'])}`",
        ]
        display(Markdown('\n'.join(summary_lines)))
    else:
        display(Markdown('**No successful trials were found yet.**'))

    for image_name in ['e0_candidate_summary.png', 'e0_step_budget_curve.png', 'e0_episode_length_ecdf.png', 'e0_success_steps_hist.png']:
        image_path = ANALYSIS_ROOT / image_name
        if image_path.exists():
            display(Image(filename=str(image_path)))
            display(Markdown(f'*Saved:* `{image_path}`'))


### E0.4 Answer field

- Recommended `max_episode_steps`: `________`
- Evidence summary: `________`
- Follow-up action: `________`
